# 🔑 Keywords Binder Manager
### Gestor de archivos `.keyb` para el Sistema de Reclutamiento
---
> Crea, edita, visualiza y elimina tus archivos de palabras clave para cada puesto de trabajo.

## Sección 0 — Instalación de dependencias

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets", "-q"])
print("✅ Dependencias listas.")

## Sección 1 — Importaciones y configuración

In [ ]:
from pathlib import Path
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

KEYB_DIR = Path("local_keywords_binder_files")
KEYB_DIR.mkdir(exist_ok=True)

print(f"✅ Listo. Directorio de Keywords Binders: {KEYB_DIR.resolve()}")

---
## Sección 2 — Crear nuevo Keywords Binder

In [ ]:
def ui_crear_keyb():
    keywords_temp = []
    out = widgets.Output()

    # --- Widgets ---
    input_kw   = widgets.Text(placeholder="Escribe una palabra clave",
                               layout=widgets.Layout(width="380px"))
    btn_add    = widgets.Button(description="Agregar",      button_style="info",    layout=widgets.Layout(width="100px"))
    btn_remove = widgets.Button(description="Quitar ultima", button_style="warning", layout=widgets.Layout(width="120px"))
    btn_clear  = widgets.Button(description="Limpiar todo", button_style="danger",  layout=widgets.Layout(width="120px"))

    lista_label = widgets.HTML("<b>Keywords agregadas:</b>")
    lista_kw    = widgets.Textarea(
        value="", placeholder="Aqui apareceran las palabras clave...",
        disabled=True, layout=widgets.Layout(width="500px", height="100px")
    )
    contador    = widgets.HTML("<i>0 palabras clave</i>")

    sep         = widgets.HTML("<hr><b>Nombre del archivo</b> (sin espacios ni extension):")
    input_name  = widgets.Text(placeholder="ej: ingeniero_de_datos",
                                layout=widgets.Layout(width="300px"))
    btn_save    = widgets.Button(description="Guardar .keyb", button_style="success", layout=widgets.Layout(width="140px"))

    def refrescar_lista():
        lista_kw.value = " ; ".join(keywords_temp)
        contador.value = f"<i>{len(keywords_temp)} palabra(s) clave</i>"

    def on_add(b):
        kw = input_kw.value.strip()
        if not kw:
            with out: print("⚠️  Escribe una palabra clave primero.")
            return
        if kw.lower() in [k.lower() for k in keywords_temp]:
            with out: print(f"⚠️  '{kw}' ya esta en la lista.")
            return
        keywords_temp.append(kw)
        input_kw.value = ""
        refrescar_lista()
        with out: clear_output()

    def on_remove(b):
        if keywords_temp:
            removed = keywords_temp.pop()
            refrescar_lista()
            with out:
                clear_output()
                print(f"Eliminada: '{removed}'")

    def on_clear(b):
        keywords_temp.clear()
        refrescar_lista()
        with out:
            clear_output()
            print("Lista limpiada.")

    def on_save(b):
        nombre = input_name.value.strip().replace(" ", "_")
        if not nombre:
            with out: print("❌ El nombre no puede estar vacio.")
            return
        if not keywords_temp:
            with out: print("❌ Agrega al menos una palabra clave.")
            return
        ruta = KEYB_DIR / f"{nombre}.keyb"
        ruta.write_text(";".join(keywords_temp), encoding="utf-8")
        with out:
            clear_output()
            print(f"✅ Guardado: {ruta}")
            print(f"   Keywords ({len(keywords_temp)}): {keywords_temp}")

    btn_add.on_click(on_add)
    btn_remove.on_click(on_remove)
    btn_clear.on_click(on_clear)
    btn_save.on_click(on_save)

    display(
        widgets.HTML("<h4>Crear nuevo Keywords Binder</h4>"),
        widgets.HBox([input_kw, btn_add, btn_remove, btn_clear]),
        lista_label, lista_kw, contador,
        sep, input_name, btn_save, out
    )

ui_crear_keyb()

---
## Sección 3 — Editar un Keywords Binder existente

In [ ]:
def ui_editar_keyb():
    out = widgets.Output()
    archivos = list(KEYB_DIR.glob("*.keyb"))

    if not archivos:
        display(widgets.HTML("<p>⚠️ No hay archivos .keyb en el directorio. Crea uno en la Seccion 2.</p>"))
        return

    # --- Selector de archivo ---
    selector = widgets.Dropdown(
        options=[f.name for f in archivos],
        description="Archivo:",
        layout=widgets.Layout(width="350px")
    )
    btn_cargar = widgets.Button(description="Cargar para editar", button_style="info", layout=widgets.Layout(width="160px"))

    # --- Area de edicion (se activa al cargar) ---
    edit_area  = widgets.Output()

    def on_cargar(b):
        ruta  = KEYB_DIR / selector.value
        texto = ruta.read_text(encoding="utf-8")
        kws   = [k.strip() for k in texto.split(";") if k.strip()]

        with edit_area:
            clear_output()
            _editor_keyb(ruta, kws)

    btn_cargar.on_click(on_cargar)

    display(
        widgets.HTML("<h4>Editar Keywords Binder existente</h4>"),
        widgets.HBox([selector, btn_cargar]),
        edit_area, out
    )


def _editor_keyb(ruta: Path, kws_inicial: list):
    """Editor interactivo para un archivo .keyb ya cargado."""
    keywords_temp = kws_inicial.copy()
    out = widgets.Output()

    display(widgets.HTML(f"<b>Editando:</b> <code>{ruta.name}</code>"))

    # Lista editable
    lista_kw  = widgets.Textarea(
        value=" ; ".join(keywords_temp),
        disabled=True,
        layout=widgets.Layout(width="500px", height="100px")
    )
    contador  = widgets.HTML(f"<i>{len(keywords_temp)} palabra(s) clave</i>")

    def refrescar():
        lista_kw.value = " ; ".join(keywords_temp)
        contador.value = f"<i>{len(keywords_temp)} palabra(s) clave</i>"

    # Agregar
    input_nueva = widgets.Text(placeholder="Nueva palabra clave", layout=widgets.Layout(width="300px"))
    btn_add     = widgets.Button(description="Agregar",       button_style="info",    layout=widgets.Layout(width="100px"))

    # Eliminar por nombre
    input_del   = widgets.Text(placeholder="Palabra clave a eliminar", layout=widgets.Layout(width="300px"))
    btn_del     = widgets.Button(description="Eliminar",      button_style="warning", layout=widgets.Layout(width="100px"))

    # Renombrar archivo
    sep2        = widgets.HTML("<hr><b>Renombrar archivo</b> (opcional, sin extension):")
    input_nuevo_nombre = widgets.Text(
        value=ruta.stem,
        layout=widgets.Layout(width="280px")
    )

    btn_save    = widgets.Button(description="Guardar cambios", button_style="success", layout=widgets.Layout(width="150px"))

    def on_add(b):
        kw = input_nueva.value.strip()
        if not kw:
            with out: print("⚠️  Escribe una palabra clave.")
            return
        if kw.lower() in [k.lower() for k in keywords_temp]:
            with out: print(f"⚠️  '{kw}' ya existe.")
            return
        keywords_temp.append(kw)
        input_nueva.value = ""
        refrescar()
        with out: clear_output()

    def on_del(b):
        kw = input_del.value.strip()
        match = next((k for k in keywords_temp if k.lower() == kw.lower()), None)
        if match:
            keywords_temp.remove(match)
            refrescar()
            input_del.value = ""
            with out:
                clear_output()
                print(f"Eliminada: '{match}'")
        else:
            with out: print(f"❌ '{kw}' no encontrada en la lista.")

    def on_save(b):
        nuevo_nombre = input_nuevo_nombre.value.strip().replace(" ", "_")
        if not nuevo_nombre:
            with out: print("❌ El nombre no puede estar vacio.")
            return
        if not keywords_temp:
            with out: print("❌ El archivo no puede quedar sin keywords.")
            return
        nueva_ruta = KEYB_DIR / f"{nuevo_nombre}.keyb"
        # Si cambio de nombre, borra el anterior
        if nueva_ruta != ruta and ruta.exists():
            ruta.unlink()
        nueva_ruta.write_text(";".join(keywords_temp), encoding="utf-8")
        with out:
            clear_output()
            print(f"✅ Guardado: {nueva_ruta}")
            print(f"   Keywords ({len(keywords_temp)}): {keywords_temp}")

    btn_add.on_click(on_add)
    btn_del.on_click(on_del)
    btn_save.on_click(on_save)

    display(
        lista_kw, contador,
        widgets.HTML("<hr><b>Agregar keyword:</b>"),
        widgets.HBox([input_nueva, btn_add]),
        widgets.HTML("<b>Eliminar keyword:</b>"),
        widgets.HBox([input_del, btn_del]),
        sep2, input_nuevo_nombre,
        btn_save, out
    )


ui_editar_keyb()

---
## Sección 4 — Ver todos los Keywords Binders

In [ ]:
def ui_ver_todos():
    archivos = list(KEYB_DIR.glob("*.keyb"))

    if not archivos:
        display(widgets.HTML("<p>⚠️ No hay archivos .keyb en el directorio.</p>"))
        return

    html = "<h4>Archivos Keywords Binder disponibles</h4>"
    html += f"<p>Directorio: <code>{KEYB_DIR.resolve()}</code></p>"
    html += "<table style='border-collapse:collapse; width:100%; font-size:13px;'>"
    html += """
        <tr style='background:#1a1a2e; color:white;'>
            <th style='padding:8px 12px; text-align:left;'>Archivo</th>
            <th style='padding:8px 12px; text-align:center;'>Keywords</th>
            <th style='padding:8px 12px; text-align:left;'>Palabras clave</th>
        </tr>
    """
    for i, f in enumerate(sorted(archivos)):
        texto = f.read_text(encoding="utf-8")
        kws   = [k.strip() for k in texto.split(";") if k.strip()]
        tags  = " ".join([f"<span style='background:#e8f4fd; border:1px solid #90caf9; border-radius:4px; padding:2px 6px; margin:2px; font-size:12px;'>{k}</span>" for k in kws])
        bg    = "#f9f9f9" if i % 2 == 0 else "#ffffff"
        html += f"""
            <tr style='background:{bg}; border-bottom:1px solid #ddd;'>
                <td style='padding:8px 12px; font-weight:bold;'>{f.name}</td>
                <td style='padding:8px 12px; text-align:center;'>{len(kws)}</td>
                <td style='padding:8px 12px;'>{tags}</td>
            </tr>
        """
    html += "</table>"
    display(HTML(html))


ui_ver_todos()

---
## Sección 5 — Subir un .keyb desde tu computadora

In [ ]:
def ui_subir_keyb():
    out = widgets.Output()
    uploader = widgets.FileUpload(accept=".keyb", multiple=False, description="Subir .keyb")
    btn      = widgets.Button(description="Procesar archivo", button_style="success")

    def on_procesar(b):
        if not uploader.value:
            with out: print("❌ No se ha subido ningun archivo.")
            return

        val = uploader.value
        if isinstance(val, dict):
            nombre  = list(val.keys())[0]
            content = bytes(list(val.values())[0]["content"])
        elif isinstance(val, (list, tuple)):
            item    = val[0]
            nombre  = item.name if hasattr(item, "name") else item["name"]
            content = bytes(item.content if hasattr(item, "content") else item["content"])

        if not nombre.endswith(".keyb"):
            with out: print("❌ El archivo debe tener extension .keyb")
            return

        content_str  = content.decode("utf-8")
        ruta_destino = KEYB_DIR / nombre
        ruta_destino.write_text(content_str, encoding="utf-8")
        kws = [k.strip() for k in content_str.split(";") if k.strip()]

        with out:
            clear_output()
            print(f"✅ Guardado: {ruta_destino}")
            print(f"   Keywords ({len(kws)}): {kws}")

    btn.on_click(on_procesar)
    display(
        widgets.HTML("<h4>Subir archivo .keyb desde tu computadora</h4>"),
        uploader, btn, out
    )


ui_subir_keyb()

---
## Sección 6 — Eliminar un Keywords Binder

In [ ]:
def ui_eliminar_keyb():
    out      = widgets.Output()
    archivos = list(KEYB_DIR.glob("*.keyb"))

    if not archivos:
        display(widgets.HTML("<p>⚠️ No hay archivos .keyb para eliminar.</p>"))
        return

    selector    = widgets.Dropdown(
        options=[f.name for f in archivos],
        description="Archivo:",
        layout=widgets.Layout(width="350px")
    )
    preview     = widgets.HTML("")
    btn_confirm = widgets.Button(description="Eliminar", button_style="danger",
                                  layout=widgets.Layout(width="120px"), disabled=True)
    checkbox    = widgets.Checkbox(value=False,
                                    description="Confirmo que deseo eliminar este archivo",
                                    layout=widgets.Layout(width="350px"))

    def actualizar_preview(change):
        ruta  = KEYB_DIR / selector.value
        texto = ruta.read_text(encoding="utf-8")
        kws   = [k.strip() for k in texto.split(";") if k.strip()]
        preview.value = f"<i style='color:#888;'>Keywords ({len(kws)}): {', '.join(kws)}</i>"

    def on_checkbox(change):
        btn_confirm.disabled = not checkbox.value

    def on_eliminar(b):
        ruta = KEYB_DIR / selector.value
        nombre = ruta.name
        ruta.unlink()
        checkbox.value        = False
        btn_confirm.disabled  = True
        # Actualizar opciones
        nuevos = [f.name for f in KEYB_DIR.glob("*.keyb")]
        selector.options = nuevos if nuevos else ["(sin archivos)"]
        with out:
            clear_output()
            print(f"🗑️  Eliminado: {nombre}")

    selector.observe(actualizar_preview, names="value")
    checkbox.observe(on_checkbox, names="value")
    btn_confirm.on_click(on_eliminar)

    # Preview inicial
    ruta  = KEYB_DIR / archivos[0].name
    texto = ruta.read_text(encoding="utf-8")
    kws   = [k.strip() for k in texto.split(";") if k.strip()]
    preview.value = f"<i style='color:#888;'>Keywords ({len(kws)}): {', '.join(kws)}</i>"

    display(
        widgets.HTML("<h4>Eliminar Keywords Binder</h4>"),
        selector, preview, checkbox, btn_confirm, out
    )


ui_eliminar_keyb()

---
## Sección 7 — Menu principal (todas las opciones en una sola celda)

In [ ]:
def menu_principal():
    out_menu = widgets.Output()

    titulo = widgets.HTML("""
        <h3>Keywords Binder Manager</h3>
        <p>Selecciona una accion:</p>
    """)

    btn_crear    = widgets.Button(description="Crear nuevo",   button_style="success", layout=widgets.Layout(width="160px"))
    btn_editar   = widgets.Button(description="Editar",        button_style="info",    layout=widgets.Layout(width="160px"))
    btn_ver      = widgets.Button(description="Ver todos",     button_style="primary", layout=widgets.Layout(width="160px"))
    btn_subir    = widgets.Button(description="Subir archivo", button_style="warning", layout=widgets.Layout(width="160px"))
    btn_eliminar = widgets.Button(description="Eliminar",      button_style="danger",  layout=widgets.Layout(width="160px"))

    def on_crear(b):
        with out_menu: clear_output(); ui_crear_keyb()
    def on_editar(b):
        with out_menu: clear_output(); ui_editar_keyb()
    def on_ver(b):
        with out_menu: clear_output(); ui_ver_todos()
    def on_subir(b):
        with out_menu: clear_output(); ui_subir_keyb()
    def on_eliminar(b):
        with out_menu: clear_output(); ui_eliminar_keyb()

    btn_crear.on_click(on_crear)
    btn_editar.on_click(on_editar)
    btn_ver.on_click(on_ver)
    btn_subir.on_click(on_subir)
    btn_eliminar.on_click(on_eliminar)

    display(
        titulo,
        widgets.HBox([btn_crear, btn_editar, btn_ver, btn_subir, btn_eliminar]),
        out_menu
    )


menu_principal()